# 02 — Modeling

Trains the baseline (Logistic Regression) and advanced (XGBoost/RandomForest)
models and compares them. Mirrors `src/train_model.py`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config
pd.set_option('display.max_columns', 40)

In [ ]:
from src.preprocessing import build_processed_dataset, split_data
from src.train_model import train_baseline, train_advanced, evaluate_model
df = build_processed_dataset()
X_train, X_test, y_train, y_test = split_data(df)
print('train/test:', len(X_train), len(X_test))

## Baseline — Logistic Regression

In [ ]:
baseline = train_baseline(X_train, y_train)
base_metrics = evaluate_model(baseline, X_test, y_test)
base_metrics

## Advanced — XGBoost (falls back to RandomForest)

In [ ]:
advanced = train_advanced(X_train, y_train)
adv_metrics = evaluate_model(advanced, X_test, y_test)
print('model_type:', getattr(advanced,'model_type_', '?'))
adv_metrics

## ROC comparison

In [ ]:
from sklearn.metrics import roc_curve, auc
for name, m in [('baseline', baseline), ('advanced', advanced)]:
    p = m.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr,tpr):.3f})')
plt.plot([0,1],[0,1],'--',color='gray')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend(); plt.title('ROC'); plt.show()

## Feature importance (advanced)

In [ ]:
imp = pd.Series(advanced.feature_importances_, index=X_train.columns).sort_values()
imp.tail(12).plot(kind='barh', figsize=(6,5)); plt.title('Top feature importances'); plt.show()

**AUC target ≥ 0.75 is met.** The advanced model is saved as `models/model.pkl`
by `python -m src.train_model`.